In [9]:
import requests
import json
import os
import time
from bs4 import BeautifulSoup
import newspaper
from newspaper import Config


In [ ]:
import os
import json
import time
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from newspaper import Article, Config
from sklearn.feature_extraction.text import TfidfVectorizer

STOP_WORDS = set(stopwords.words('english'))

# --- Helper Functions ---

def load_saved_links(file_path):
    """Load saved links from a JSON array file."""
    saved_links = set()
    
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if not content:
                    print(f"Warning: {file_path} is empty. Initializing new empty list.")
                    return saved_links
                data = json.loads(content)
                for entry in data:
                    if 'link' in entry:
                        saved_links.add(entry['link'])
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON file: {e}")
            print(f"Reinitializing {file_path} to empty array.")
            # Optionally reinitialize the file if you want
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump([], f)
            saved_links = set()
    return saved_links


def fetch_search_results(url, headers, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except requests.exceptions.RequestException as e:
            print(f"Error fetching search results: {e}")
            time.sleep(2)
    return None

def extract_results(soup, limit=10):
    """Extract a limited number of search result links."""
    return soup.find_all('a', class_='title', limit=limit) if soup else []

def extract_article_with_newspaper(link, user_agent):
    """Extract article content and summary using Newspaper3k."""
    try:
        config = Config()
        config.browser_user_agent = user_agent
        article = Article(link, config=config)
        article.download()
        article.parse()
        text = article.text
        summary = ' '.join(text.split()[:50])
        publish_date = article.publish_date
        publish_date_str = publish_date.strftime("%Y-%m-%d %H:%M:%S") if publish_date else "Unknown"
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Error extracting article with Newspaper: {e}")
        time.sleep(2)
        return None, None, None

def fallback_extraction(link, headers):
    """Fallback extraction with BeautifulSoup."""
    try:
        response = requests.get(link, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        article_div = soup.find('div', class_='article-body') or soup.find('div', class_='content')
        text = article_div.get_text(separator=' ', strip=True) if article_div else soup.get_text(separator=' ', strip=True)
        summary = ' '.join(text.split()[:50])
        publish_date_str = "Unknown"
        return text, summary, publish_date_str
    except Exception as e:
        print(f"Fallback extraction failed for {link}: {e}")
        return None, None, None

def extract_article(link, user_agent, headers):
    """Try Newspaper extraction first, fallback to BeautifulSoup if needed."""
    text, summary, publish_date = extract_article_with_newspaper(link, user_agent)
    if not text or len(text.split()) < 100:
        print("Falling back to BeautifulSoup extraction.")
        text, summary, publish_date = fallback_extraction(link, headers)
    return text, summary, publish_date

def extract_keywords(text, top_n=10):
    """Smart keyword extraction using TF-IDF."""
    if not text:
        return []
    vectorizer = TfidfVectorizer(stop_words='english', max_features=50)
    tfidf_matrix = vectorizer.fit_transform([text.lower()])
    feature_array = vectorizer.get_feature_names_out()
    tfidf_scores = tfidf_matrix.toarray().flatten()
    sorted_indices = tfidf_scores.argsort()[::-1]
    keywords = [feature_array[idx] for idx in sorted_indices[:top_n]]
    return keywords

def save_to_json(data, file_path):
    """Save the provided data into a JSON file as a JSON array."""
    try:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    existing_data = json.load(f)
                except json.JSONDecodeError:
                    existing_data = []
        else:
            existing_data = []

        existing_data.append(data)

        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(existing_data, f, indent=4, ensure_ascii=False)
    except IOError as e:
        print(f"Error saving to file: {e}")

# --- Main Workflow ---

def main():
    # Define search terms
    search_terms = ["Credential Stuffing"]

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }
    user_agent = headers['User-Agent']
    file_path = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/BingSearchData.json'

    saved_links = load_saved_links(file_path)
    print(f"Loaded {len(saved_links)} saved links from previous runs.")

    desired_valid_articles = 10

    for term in search_terms:
        print(f"\nSearching for: {term}")
        search_url = f"https://www.bing.com/news/search?q={term}"
        soup = fetch_search_results(search_url, headers)
        candidate_links = extract_results(soup, limit=20)

        valid_articles_count = 0
        seen_links = set()

        for result in candidate_links:
            link = result.get('href')
            if link in seen_links or link in saved_links:
                continue
            seen_links.add(link)

            title = result.get_text(strip=True)
            print(f"Title: {title}")
            print(f"Link: {link}")
            timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

            content, summary, publish_date = extract_article(link, user_agent, headers)

            if content and len(content.split()) >= 100:
                print(f"Summary: {summary}")

                keywords = extract_keywords(summary)

                data = {
                    'search_term': term,
                    'timestamp': timestamp,
                    'title': title,
                    'link': link,
                    'content': content,
                    'summary': summary,
                    'keywords': keywords,
                    'publish_date': publish_date
                }

                try:
                    save_to_json(data, file_path)
                    saved_links.add(link)
                    valid_articles_count += 1
                except (TypeError, ValueError) as e:
                    print(f"Error saving data for link {link}: {e}")

                print(f"Valid articles so far: {valid_articles_count}/{desired_valid_articles}")

                if valid_articles_count >= desired_valid_articles:
                    break
            else:
                print("Excluding this link due to insufficient extracted content.")

        if valid_articles_count < desired_valid_articles:
            print(f"{valid_articles_count} valid articles found for '{term}'.")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jaytlinaskew/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loaded 0 saved links from previous runs.

Searching for: Credential Stuffing
Title: What is credential stuffing?
Link: https://www.businessinsider.com/personal-finance/credit-score/what-is-credential-stuffing
Summary: This story is available exclusively to Business Insider subscribers. Become an Insider and start reading now. Keeping a list of diverse usernames and passwords can be difficult to manage, especially for those with bad memories or an aversion to organizing. It's far easier for some of us to keep one
Valid articles so far: 1/10
Title: Exposing ATO Blind Spots: How Fraudsters Are Outsmarting Traditional Defenses
Link: https://www.forbes.com/councils/forbestechcouncil/2025/04/25/exposing-ato-blind-spots-how-fraudsters-are-outsmarting-traditional-defenses/
Summary: Nick Rieniets, Kasada CTO, stopping bot attacks others can't with novel mitigation techniques that beat cybercriminals at their own game. getty For years, security teams have been playing Whac-A-Mole with credential